# Protocol Explorer — How Protocols Actually Work

Beneath-the-surface mechanics of what happens inside `https://google.com`.

**Not repeated here:** protocol definitions, TCP vs UDP table, port numbers, DNS basics — [networking_fundamentals_1](#notebook-4479634647434063) has them.

**Flow:** the full story → each protocol's one job → protocols use each other (the stack) → TCP handshake → how TCP counts bytes (SEQ/LEN/ACK) → packet loss & recovery → UDP's philosophy → TLS → recap

## 1. The Full Story — `https://google.com`, One Step per Protocol

* **DNS** finds the IP address
* **IP** routes packets to the server
* **TCP** establishes a reliable connection (handshake)
* **TLS** creates an encrypted connection (HTTPS only)
* **HTTP** request sent → response received
* Browser renders the webpage

**One-liner:** DNS finds the address, IP routes packets, TCP guarantees delivery, TLS encrypts, HTTP transfers the content.

## 2. Each Protocol's One Job — and What IP Does NOT Do

| Protocol | Its one job |
| --- | --- |
| DNS | name → IP address |
| IP | deliver packets from source to destination — addressing + routing |
| TCP | make delivery reliable + ordered |
| TLS | encrypt the conversation |
| HTTP | carry the web request/response |

**IP is best-effort:** it only cares WHERE the packet goes. It does NOT handle:

* lost packets
* ordering
* reliability

All of that is **TCP's job** — IP and TCP are a team: IP moves, TCP guarantees.

## 3. Protocols Use Each Other — the Stack

* Big insight: protocols don't work alone — **high-level protocols USE lower-level protocols**
* HTTP/HTTPS don't deliver anything themselves — they hand their data to TCP, and TCP hands packets to IP

**The stack for a website**

```
HTTP / HTTPS  → what to say (the content)
  uses TCP    → makes it arrive, complete and in order
    uses IP   → actually moves the packets
```

* Each protocol only trusts the one below it — that's why it's called a **stack**
* Another example: DNS uses UDP — a tiny lookup doesn't need TCP's bookkeeping
* This is the same idea as the TCP/IP layers in [networking_fundamentals_1](#notebook-4479634647434063) — "layers" literally means which protocol uses which

## 4. TCP Three-Way Handshake — Before Any Data

1. Client → **SYN** (Synchronize) — "Can we talk?"
2. Server → **SYN + ACK** — "I heard you. Can you hear me?"
3. Client → **ACK** (Acknowledgement) — "Yes, I hear you too."

→ Connection established, data transfer begins.

* **SYN** = "I want to start a connection"
* **ACK** = "I received your message"

## 5. How TCP Counts Every Byte — SEQ, LEN, ACK

* **SEQ (sequence number)** — the byte number this data starts from; increases as data flows
* **LEN** — how many bytes of data are in this packet
* **ACK number** — the next byte the receiver expects

**Worked example (remember this one)**

* Sender: `Seq = 100`, `Len = 78` → "78 bytes, starting at byte 100"
* Receiver replies: `Ack = 178` → "I got bytes 100–177. Send me byte 178 next."

**Formula: next ACK = SEQ + LEN**

* This counting is HOW TCP guarantees order — every byte is tracked and confirmed

## 6. When a Packet Is Lost — the Recovery Chain

1. **Packet loss** — 3 packets sent, only 2 arrive (e.g. packet 2 missing)
2. **Duplicate ACK** — receiver keeps repeating the same ACK ("I still need packet 2" — same number, 3 times)
3. **Retransmission** — sender sees the repeated ACKs, realizes a packet is missing, **sends it again**

* This chain is why TCP is called **reliable** — loss is detected AND recovered automatically, no application help needed

## 7. UDP's Philosophy — the Opposite Bet

* "Sent it. Hope it arrives."
* No handshake, no ACK, no retransmission, no ordering — ALL the TCP bookkeeping is skipped
* Result: fast + lightweight
* Used where speed beats perfection: DNS, gaming, video streaming, voice calls

* Connect: TCP's reliability = handshake + byte counting + retransmission; UDP = skip all three → speed

## 8. TLS — Making the Connection Private

* **HTTPS = HTTP + TLS** — plain HTTP flowing through an encrypted pipe
* Order: TCP handshake connects the two sides → TLS handshake secures the channel → then HTTP flows encrypted

**TLS handshake — 3 things happen**

1. Server's **certificate** is exchanged — proof of identity
2. **Encryption method** (cipher) is selected
3. **Encryption keys** are created

After that, every request and response is encrypted.

* Quick recall: HTTP = port 80, plain text | HTTPS = port 443, TLS-encrypted

## 9. Quick Recap — Connect the Dots

* **google.com story:** DNS (find IP) → IP (route, best-effort) → TCP (SYN / SYN-ACK / ACK, then counts every byte: SEQ + LEN) → TLS (cert, cipher, keys) → HTTP (content) → render
* **The stack:** high-level protocols use lower-level ones — HTTP/HTTPS use TCP, TCP uses IP, DNS uses UDP
* **TCP reliability** = handshake + byte counting (`next ACK = SEQ + LEN`) + duplicate ACK → retransmission
* **UDP** = skip all of that → speed
* **TLS** = the encryption layer that turns HTTP into HTTPS
* **IP vs TCP division:** IP moves packets, TCP makes them arrive — complete and in order